# Cats vs Dogs — Binary Classification CNN
**GPU recommandé : Runtime > Change runtime type > T4 GPU**

## 0. Téléchargement du dataset (tensorflow_datasets)

In [ ]:
import tensorflow_datasets as tfds
import tensorflow as tf
import numpy as np
import os
from PIL import Image

print('Téléchargement cats_vs_dogs...')
dataset, info = tfds.load('cats_vs_dogs', with_info=True, as_supervised=True)
full_ds = dataset['train']
print(f'Total images disponibles: {info.splits["train"].num_examples}')

os.makedirs('data/cats_dogs/train/train/cat', exist_ok=True)
os.makedirs('data/cats_dogs/train/train/dog', exist_ok=True)
os.makedirs('data/cats_dogs/test/test', exist_ok=True)

N_TRAIN = 8000
N_TEST  = 1000
label_names = {0: 'cat', 1: 'dog'}
counts = {'cat': 0, 'dog': 0}
test_count = 0
total = 0

for image, label in full_ds:
    if total >= N_TRAIN + N_TEST:
        break
    cls = label_names[int(label)]
    img_pil = Image.fromarray(image.numpy())
    if total < N_TRAIN:
        path = f'data/cats_dogs/train/train/{cls}/{cls}.{counts[cls]}.jpg'
        img_pil.save(path)
        counts[cls] += 1
    else:
        path = f'data/cats_dogs/test/test/{test_count}.jpg'
        img_pil.save(path)
        test_count += 1
    total += 1
    if total % 1000 == 0:
        print(f'  {total}/{N_TRAIN + N_TEST} images...')

print(f'Dataset pret: {counts} | test: {test_count}')


## 1. Data Loading & Generators

In [ ]:
import os, re
from glob import glob
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split

np.random.seed(42); tf.random.set_seed(42)

DATA_ROOT = Path('data/cats_dogs')
train_dir = (DATA_ROOT/'train'/'train') if (DATA_ROOT/'train'/'train').exists() else (DATA_ROOT/'train')
test_dir  = (DATA_ROOT/'test'/'test')   if (DATA_ROOT/'test'/'test').exists()   else (DATA_ROOT/'test')

IMG_HEIGHT, IMG_WIDTH = 128, 128
BATCH_SIZE = 32
SEED = 1337

def build_df(folder, labeled=True):
    files = []
    for ex in ('*.jpg','*.jpeg','*.png','*.bmp'):
        files.extend(glob(str(folder/'**'/ex), recursive=True))
    if not files:
        raise FileNotFoundError(f'No images under {folder}')
    rows = []
    for f in files:
        if labeled:
            name   = Path(f).name.lower()
            parent = Path(f).parent.name.lower()
            if parent in {'cat','cats'}:   label = 'cat'
            elif parent in {'dog','dogs'}: label = 'dog'
            else:
                if re.search(r'(^|[^a-z])cat([^a-z]|$)', name):   label = 'cat'
                elif re.search(r'(^|[^a-z])dog([^a-z]|$)', name): label = 'dog'
                else: continue
            rows.append({'filepath': f, 'label': label})
        else:
            rows.append({'filepath': f})
    return pd.DataFrame(rows)

df_train_full = build_df(train_dir, labeled=True)
df_test_full  = build_df(test_dir,  labeled=False)
df_tr, df_val = train_test_split(df_train_full, test_size=0.2,
                                  stratify=df_train_full['label'], random_state=SEED)

train_gen = ImageDataGenerator(rescale=1./255, rotation_range=30,
                                width_shift_range=0.15, height_shift_range=0.15,
                                zoom_range=0.3, horizontal_flip=True,
                                brightness_range=[0.8, 1.2])
val_gen  = ImageDataGenerator(rescale=1./255)
test_gen = ImageDataGenerator(rescale=1./255)

train_flow = train_gen.flow_from_dataframe(df_tr, x_col='filepath', y_col='label',
    target_size=(IMG_HEIGHT,IMG_WIDTH), class_mode='binary',
    batch_size=BATCH_SIZE, shuffle=True, seed=SEED, validate_filenames=False)

val_flow = val_gen.flow_from_dataframe(df_val, x_col='filepath', y_col='label',
    target_size=(IMG_HEIGHT,IMG_WIDTH), class_mode='binary',
    batch_size=BATCH_SIZE, shuffle=False, validate_filenames=False)

test_flow = test_gen.flow_from_dataframe(df_test_full, x_col='filepath', y_col=None,
    target_size=(IMG_HEIGHT,IMG_WIDTH), class_mode=None,
    batch_size=BATCH_SIZE, shuffle=False, validate_filenames=False)

print(f'Train:{train_flow.samples} | Val:{val_flow.samples} | Test:{test_flow.samples}')
print(f'Classes: {train_flow.class_indices}')


## 2. Inspection des données

In [ ]:
import matplotlib.pyplot as plt

label_counts = df_tr['label'].value_counts()
print('=== Repartition classes (train) ===')
print(label_counts)
print(f'Ratio cat/dog: {label_counts["cat"]/label_counts["dog"]:.3f}')

fig, axes = plt.subplots(3, 6, figsize=(16, 8))
batch_images, batch_labels = next(train_flow)
idx_to_class = {v: k for k, v in train_flow.class_indices.items()}

for i, ax in enumerate(axes.flat):
    if i >= len(batch_images): break
    ax.imshow(batch_images[i])
    cls = idx_to_class[int(batch_labels[i])]
    color = '#6C3FC5' if cls == 'cat' else '#F5A623'
    ax.set_title(cls, fontsize=10, fontweight='bold', color=color)
    ax.axis('off')

plt.suptitle("Echantillon d'images augmentees", fontsize=14)
plt.tight_layout()
plt.savefig('sample_grid.png', dpi=100, bbox_inches='tight')
plt.show()


## 3 & 4. Architecture CNN + Optimisation

In [ ]:
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

def build_cnn(input_shape=(128, 128, 3), dropout_rate=0.4):
    model = models.Sequential([
        layers.Conv2D(32, (3,3), activation='relu', padding='same', input_shape=input_shape),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.2),

        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.25),

        layers.Conv2D(128, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(128, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.3),

        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(dropout_rate),
        layers.Dense(1, activation='sigmoid')
    ], name='cats_vs_dogs_cnn')
    return model

model = build_cnn(input_shape=(IMG_HEIGHT, IMG_WIDTH, 3))
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)
model.summary()


## 5. Entraînement

In [ ]:
EPOCHS = 30

callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1),
    ModelCheckpoint('best_model.keras', monitor='val_accuracy', save_best_only=True, verbose=1)
]

history = model.fit(
    train_flow, epochs=EPOCHS,
    validation_data=val_flow,
    callbacks=callbacks, verbose=1
)
print(f'Termine en {len(history.history["loss"])} epochs')


In [ ]:
def plot_history(hist, title=''):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ep = range(1, len(hist['loss'])+1)
    ax1.plot(ep, hist['loss'],     color='#6C3FC5', lw=2, label='Train')
    ax1.plot(ep, hist['val_loss'], color='#F5A623', lw=2, ls='--', label='Val')
    ax1.set_title(f'Loss {title}'); ax1.set_xlabel('Epoch'); ax1.legend(); ax1.grid(alpha=0.3)
    ax2.plot(ep, hist['accuracy'],     color='#6C3FC5', lw=2, label='Train')
    ax2.plot(ep, hist['val_accuracy'], color='#F5A623', lw=2, ls='--', label='Val')
    ax2.set_title(f'Accuracy {title}'); ax2.set_xlabel('Epoch'); ax2.legend(); ax2.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'curves_{title.replace(" ","_")}.png', dpi=100, bbox_inches='tight')
    plt.show()

plot_history(history.history, 'augmente')


## 6. Évaluation — Confusion Matrix & Rapport

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

val_flow.reset()
y_pred_prob = model.predict(val_flow, verbose=1)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()
y_true = val_flow.labels

val_loss, val_acc = model.evaluate(val_flow, verbose=0)
print(f'Val Loss: {val_loss:.4f} | Val Accuracy: {val_acc:.4f}')

idx_to_class = {v: k for k, v in train_flow.class_indices.items()}
target_names = [idx_to_class[0], idx_to_class[1]]
print(classification_report(y_true, y_pred, target_names=target_names))

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm, display_labels=target_names).plot(ax=ax, cmap='Purples', colorbar=False)
ax.set_title('Matrice de confusion')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=100, bbox_inches='tight')
plt.show()


## 7. Inférence sur le test set

In [ ]:
test_flow.reset()
test_probs = model.predict(test_flow, verbose=1).flatten()

THRESHOLD = 0.5
test_labels = ['dog' if p > THRESHOLD else 'cat' for p in test_probs]

df_preds = pd.DataFrame({
    'filepath':   test_flow.filenames,
    'prob_dog':   test_probs.round(4),
    'pred_label': test_labels
})
df_preds.to_csv('test_predictions.csv', index=False)
print(f'{len(df_preds)} predictions sauvegardees dans test_predictions.csv')
print(df_preds.head(10))
print(df_preds['pred_label'].value_counts())


## 8. Baseline (sans augmentation) vs Augmenté

In [ ]:
base_gen = ImageDataGenerator(rescale=1./255)
base_flow = base_gen.flow_from_dataframe(df_tr, x_col='filepath', y_col='label',
    target_size=(IMG_HEIGHT,IMG_WIDTH), class_mode='binary',
    batch_size=BATCH_SIZE, shuffle=True, seed=SEED, validate_filenames=False)

baseline_model = build_cnn(input_shape=(IMG_HEIGHT, IMG_WIDTH, 3))
baseline_model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                        loss='binary_crossentropy', metrics=['accuracy'])

history_base = baseline_model.fit(
    base_flow, epochs=EPOCHS, validation_data=val_flow,
    callbacks=[EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)],
    verbose=1
)

base_val_acc = max(history_base.history['val_accuracy'])
aug_val_acc  = max(history.history['val_accuracy'])
print(f'Baseline : {base_val_acc:.4f}')
print(f'Augmente : {aug_val_acc:.4f}')
print(f'Gain     : +{(aug_val_acc - base_val_acc)*100:.2f}%')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, hist, label, color in zip(axes,
    [history_base.history, history.history],
    ['Baseline', 'Augmente'], ['#E74C3C', '#6C3FC5']):
    ep = range(1, len(hist['loss'])+1)
    ax.plot(ep, hist['accuracy'],     color=color, lw=2, label='Train')
    ax.plot(ep, hist['val_accuracy'], color=color, lw=2, ls='--', label='Val')
    ax.set_title(f'{label} — Accuracy')
    ax.legend(); ax.grid(alpha=0.3); ax.set_ylim(0.5, 1.0)
plt.tight_layout()
plt.savefig('baseline_vs_augmented.png', dpi=100, bbox_inches='tight')
plt.show()


## 9. Gestion du déséquilibre de classes

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(train_flow.labels)
weights = compute_class_weight('balanced', classes=classes, y=train_flow.labels)
class_weights = dict(zip(classes.astype(int), weights))
print(f'Class weights: {class_weights}')

counts = df_tr['label'].value_counts()
is_imbalanced = abs(counts.max() - counts.min()) > counts.sum() * 0.05
print(f'Classes desequilibrees: {is_imbalanced}')

if is_imbalanced:
    print('Reentrainement avec class_weight...')
    m2 = build_cnn(input_shape=(IMG_HEIGHT, IMG_WIDTH, 3))
    m2.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
               loss='binary_crossentropy', metrics=['accuracy'])
    train_flow.reset()
    m2.fit(train_flow, epochs=EPOCHS, validation_data=val_flow,
           class_weight=class_weights,
           callbacks=[EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)],
           verbose=1)
else:
    print('Classes equilibrees, class_weight non necessaire.')


## 10. Sauvegarde du modèle et config

In [ ]:
import json

model.save('cats_dogs_model.keras')
print('Modele sauvegarde: cats_dogs_model.keras')

config = {
    'model_name': 'cats_vs_dogs_cnn',
    'img_size': [IMG_HEIGHT, IMG_WIDTH],
    'batch_size': BATCH_SIZE,
    'epochs_trained': len(history.history['loss']),
    'optimizer': 'Adam',
    'learning_rate': 1e-3,
    'loss': 'binary_crossentropy',
    'dropout_rate': 0.4,
    'augmentation': {'rotation': 30, 'shift': 0.15, 'zoom': 0.3,
                     'horizontal_flip': True, 'brightness': [0.8, 1.2]},
    'val_accuracy': round(float(max(history.history['val_accuracy'])), 4),
    'val_loss':     round(float(min(history.history['val_loss'])),     4),
    'class_indices': train_flow.class_indices
}
with open('training_config.json', 'w') as f:
    json.dump(config, f, indent=2)
print('Config sauvegardee: training_config.json')
print(json.dumps(config, indent=2))


## 11. Extension — Transfer Learning MobileNetV2

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import Model, Input

base_model = MobileNetV2(input_shape=(IMG_HEIGHT,IMG_WIDTH,3),
                          include_top=False, weights='imagenet')
base_model.trainable = False

inputs  = Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3))
x       = base_model(inputs, training=False)
x       = layers.GlobalAveragePooling2D()(x)
x       = layers.Dense(128, activation='relu')(x)
x       = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)

tl_model = Model(inputs, outputs, name='mobilenetv2_cats_dogs')
tl_model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss='binary_crossentropy', metrics=['accuracy'])

print('Phase 1: entrainement de la tete uniquement')
train_flow.reset()
history_tl = tl_model.fit(
    train_flow, epochs=10, validation_data=val_flow,
    callbacks=[EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)],
    verbose=1
)

print('Phase 2: fine-tuning des 20 dernieres couches')
base_model.trainable = True
for layer in base_model.layers[:-20]:
    layer.trainable = False
tl_model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
                  loss='binary_crossentropy', metrics=['accuracy'])
train_flow.reset()
history_ft = tl_model.fit(
    train_flow, epochs=10, validation_data=val_flow,
    callbacks=[EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)],
    verbose=1
)

tl_acc = max(history_ft.history['val_accuracy'])
print(f'MobileNetV2 val_accuracy : {tl_acc:.4f}')
print(f'CNN scratch  val_accuracy: {max(history.history["val_accuracy"]):.4f}')


## 12. Récapitulatif final

In [ ]:
print('='*55)
print('         RECAP DES EXPERIENCES')
print('='*55)
print(f'  Baseline (sans augmentation) : {max(history_base.history["val_accuracy"]):.4f}')
print(f'  CNN + augmentation           : {max(history.history["val_accuracy"]):.4f}')
print(f'  MobileNetV2 (transfer)       : {tl_acc:.4f}')
print('='*55)
print()
print('Fichiers produits:')
for f in ['best_model.keras','cats_dogs_model.keras',
          'training_config.json','test_predictions.csv',
          'sample_grid.png','confusion_matrix.png',
          'baseline_vs_augmented.png']:
    exists = 'OK' if os.path.exists(f) else 'manquant'
    print(f'  {f:<35} [{exists}]')
